# V4 KOLF → VCC 2026 controls-final 预测与提交打包

使用已经训练好的 `KOLF_V4_delta_model.pt`，不重新训练。对每个 VCC context：

1. 从官方 control 细胞抽取 400 个作为基线；
2. 将 control 转换到与 KOLF 训练一致的 `normalize_total + log1p` 空间；
3. 模型预测 KOLF 输出基因上的 `delta`，并计算 `max(0, VCC_control_log + delta)`；
4. 模型未覆盖的官方基因保持 VCC control 表达（delta=0）；
5. 逆变换为整数 raw counts，稀疏化并合并三个 context；
6. 本地检查提交格式，可调用官方 `vcc prep` 生成 `.vcc`。

最终提交形状：300 targets × 3 contexts × 400 cells = 360,000 cells，18,533 genes。  
官方格式说明：https://virtualcellchallenge.org/evaluation  
CLI 说明：https://vcc-cli-wiki.virtualcellchallenge.org/

**科学限制**：KOLF 单细胞类型模型直接迁移到 D/E/F，且大量 VCC 靶点没有 KOLF 直接监督；本流程首先保证“当前模型真实参与、格式可提交”，不保证排行榜表现。

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import h5py
import torch
import torch.nn as nn
from scipy.sparse import csr_matrix, csc_matrix, issparse, vstack

# =============================================================================
# Paths：只需确认这里指向你当前 V4 文件和 controls-final 文件夹
# =============================================================================
PROJECT_DIR = Path(r"E:\VSCODE\PrePR-CT\Cell-Type-Specific-Graphs-main")
VCC_DIR = Path(r"E:\VSCODE\Virtual Cell Challenge\Original_datacc_2026_controls_final")

DATA_DIR = PROJECT_DIR / "Data"
GRAPH_DIR = PROJECT_DIR / "graphs" / "KOLF_V4"
MODEL_PATH = PROJECT_DIR / "model_checkpoints" / "KOLF_V4_delta_model.pt"

GRAPH_INPUT_PATH = DATA_DIR / "KOLF_V4_graph_input.h5ad"
FULL_INPUT_PATH = DATA_DIR / "KOLF_V4_full_QC.h5ad"
OUTPUT_ORDER_PATH = DATA_DIR / "V4_output_gene_order.csv"
OOD_SPLIT_PATH = DATA_DIR / "V4_ood_split.json"

GENE_NAMES_PATH = VCC_DIR / "gene_names.csv"
PERT_COUNTS_PATH = VCC_DIR / "pert_counts.csv"
MANIFEST_PATH = VCC_DIR / "manifest.json"
OUTPUT_H5AD = VCC_DIR / "KOLF_V4_prediction.h5ad"
OUTPUT_VCC = VCC_DIR / "KOLF_V4_prediction.vcc"
TARGET_REPORT_PATH = VCC_DIR / "KOLF_V4_prediction_target_report.csv"
RUN_MANIFEST_PATH = VCC_DIR / "KOLF_V4_prediction_manifest.json"

# =============================================================================
# Prediction configuration
# =============================================================================
CONTEXTS = ["D", "E", "F"]
CELLS_PER_TARGET = 400
PREDICT_BATCH_SIZE = 128
MAX_GENES_PER_CELL = 1000       # 每细胞最多保留的预测基因；实际只存储非零 counts。
MAX_STORED_ENTRIES = 4_750_000_000
MAX_COUNTS_PER_CELL = 1_000_000
RANDOM_SEED = 42
RUN_VCC_PREP = False            # H5AD 检查通过后改为 True，运行最后单元生成 .vcc。

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
rng = np.random.RandomState(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from model import GNN

required_paths = [
    PROJECT_DIR, VCC_DIR, GRAPH_DIR, MODEL_PATH,
    GRAPH_INPUT_PATH, FULL_INPUT_PATH, OUTPUT_ORDER_PATH, OOD_SPLIT_PATH,
    GENE_NAMES_PATH, PERT_COUNTS_PATH, MANIFEST_PATH,
] + [VCC_DIR / f"context_{c}.h5ad" for c in CONTEXTS]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"找不到：{path}")

print("AnnData:", ad.__version__)
print("Device:", device)
print("Model:", MODEL_PATH)
print("Output:", OUTPUT_H5AD)
print("Max genes stored per cell:", MAX_GENES_PER_CELL)

In [ ]:
# =============================================================================
# Official panel + V4 files
# gene_names.csv 官方说明为 headerless CSV，不能直接用默认 header=0。
# =============================================================================
def read_official_genes(path):
    df = pd.read_csv(path, header=None)
    genes = df.iloc[:, 0].astype(str).str.strip().tolist()
    if genes and genes[0].lower() in {"gene", "gene_name", "gene_symbol", "genes"}:
        genes = genes[1:]
    return genes

official_genes = read_official_genes(GENE_NAMES_PATH)
if len(official_genes) != 18533 or len(set(official_genes)) != 18533:
    raise ValueError(f"官方基因应为 18,533 个唯一符号，实际为 {len(official_genes)} / {len(set(official_genes))}")
official_gene_to_pos = {gene: i for i, gene in enumerate(official_genes)}

pert_df = pd.read_csv(PERT_COUNTS_PATH)
pert_col = "target_gene" if "target_gene" in pert_df.columns else pert_df.columns[0]
target_genes = pert_df[pert_col].astype(str).str.strip().tolist()
if len(target_genes) != 300 or len(set(target_genes)) != 300:
    raise ValueError(f"官方靶点应为 300 个唯一符号，实际为 {len(target_genes)} / {len(set(target_genes))}")
if "non-targeting" in set(target_genes):
    raise ValueError("pert_counts.csv 中不应包含 non-targeting 提交行")

with open(OOD_SPLIT_PATH, encoding="utf-8") as handle:
    ood_split_info = json.load(handle)
training_targets = [str(g) for g in ood_split_info.get("training_targets", [])]
testing_genes = [str(g) for g in ood_split_info.get("testing_genes", [])]

output_order_df = pd.read_csv(OUTPUT_ORDER_PATH)
output_genes = output_order_df.gene_name.astype(str).tolist()
if not output_order_df.output_index.is_unique:
    raise ValueError("V4_output_gene_order.csv 的 output_index 不唯一")
missing_output_in_official = [g for g in output_genes if g not in official_gene_to_pos]
if missing_output_in_official:
    raise ValueError(f"V4 输出基因不在官方面板，示例：{missing_output_in_official[:20]}")
output_official_positions = np.array(
    [official_gene_to_pos[g] for g in output_genes], dtype=np.int32
)
output_gene_set = set(output_genes)

print("Official genes:", len(official_genes))
print("Official targets:", len(target_genes))
print("V4 model output genes:", len(output_genes))
print("Official genes with delta=0:", len(official_genes) - len(output_genes))
print("Official targets with KOLF output-gene features:", sum(g in output_gene_set for g in target_genes))
print("KOLF direct-training targets in VCC panel:", len(set(target_genes) & set(training_targets)))
print("KOLF OOD targets in VCC panel:", len(set(target_genes) & set(testing_genes)))

In [ ]:
# =============================================================================
# 与 V4 Training 完全一致的模型结构和 control-derived target features
# =============================================================================
def matrix_mean_var(X):
    """稀疏矩阵也直接计算逐基因均值和总体方差，不 densify 全部 control。"""
    if issparse(X):
        mean = np.asarray(X.mean(axis=0)).ravel()
        mean_square = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    else:
        X = np.asarray(X)
        mean = X.mean(axis=0)
        mean_square = np.square(X).mean(axis=0)
    var = np.maximum(mean_square - np.square(mean), 0)
    return mean.astype(np.float32), var.astype(np.float32)


def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


def build_control_gene_features(feature_adata, feature_dim=124):
    """返回训练特征，以及缺失靶点 fallback 所需的原始统计缩放参数。"""
    feature_ctrl = feature_adata[feature_adata.obs["condition"] == "control"].copy()
    if len(feature_ctrl) < 10:
        raise ValueError("KOLF control 细胞不足，无法构建靶点特征")
    n_pcs = min(50, feature_dim - 4, feature_ctrl.n_obs - 1, feature_ctrl.n_vars - 1)
    sc.pp.pca(feature_ctrl, n_comps=n_pcs, random_state=RANDOM_SEED)

    features = np.zeros((feature_adata.n_vars, feature_dim), dtype=np.float32)
    pcs = np.asarray(feature_ctrl.varm["PCs"], dtype=np.float32)
    features[:, :n_pcs] = (
        (pcs - pcs.mean(axis=0, keepdims=True)) / (pcs.std(axis=0, keepdims=True) + 1e-6)
    )

    mean_expr, var_expr = matrix_mean_var(feature_ctrl.X)
    if issparse(feature_ctrl.X):
        detection = np.asarray((feature_ctrl.X > 0).sum(axis=0)).ravel() / feature_ctrl.n_obs
    else:
        detection = (np.asarray(feature_ctrl.X) > 0).mean(axis=0)
    log_var = np.log1p(var_expr)

    features[:, -4] = (mean_expr - mean_expr.mean()) / (mean_expr.std() + 1e-6)
    features[:, -3] = (log_var - log_var.mean()) / (log_var.std() + 1e-6)
    features[:, -2] = detection.astype(np.float32)
    features[:, -1] = 1.0
    scaling = {
        "mean_center": float(mean_expr.mean()),
        "mean_scale": float(mean_expr.std() + 1e-6),
        "logvar_center": float(log_var.mean()),
        "logvar_scale": float(log_var.std() + 1e-6),
    }
    return features, scaling


class DeltaPredictor(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)


graph_adata = sc.read(GRAPH_INPUT_PATH)
full_adata_all = sc.read(FULL_INPUT_PATH)
if not graph_adata.obs_names.equals(full_adata_all.obs_names):
    raise ValueError("graph_input 与 full_QC 细胞顺序不一致")
full_adata = full_adata_all[:, output_genes].copy()
del full_adata_all

# 训练时的 target feature 来自 full control + output gene panel，保持同样顺序与随机种子。
full_gene_features, feature_scaling = build_control_gene_features(full_adata, feature_dim=124)
full_gene_to_index = {g: i for i, g in enumerate(full_adata.var_names.astype(str))}
kolf_target_features = {
    gene: full_gene_features[full_gene_to_index[gene]].copy()
    for gene in target_genes if gene in full_gene_to_index
}

# 单张 V4 coexpression graph。
graph_files = sorted(GRAPH_DIR.glob("*_coexpr_graph.pkl"))
if len(graph_files) != 1:
    raise ValueError(f"{GRAPH_DIR} 应恰好有 1 张图，实际为 {len(graph_files)} 张")
graph = torch.load(graph_files[0], map_location=device)
graph.x = graph.x.float()
graph.pos = graph.pos.long()
graph_gene_indices = graph.pos.tolist()
if graph.pos.numel() and (
    int(graph.pos.min()) < 0 or int(graph.pos.max()) >= graph_adata.n_vars
):
    raise ValueError("graph.pos 超出 graph_input 基因范围")
cell_type_key = str(graph_adata.obs.cell_type.iloc[0])

# KOLF control 均值用于 VCC 缺失 graph-input genes 的补齐；节点均值/方差作为 fallback。
kolf_ctrl_graph = graph_adata[graph_adata.obs.condition == "control"]
kolf_graph_mean_all, kolf_graph_var_all = matrix_mean_var(kolf_ctrl_graph.X)
kolf_node_mean = kolf_graph_mean_all[graph_gene_indices]
kolf_node_var = kolf_graph_var_all[graph_gene_indices]

# 从 checkpoint 的 perturbation embedding 推断 num_perts，避免训练/预测不一致。
checkpoint = torch.load(MODEL_PATH, map_location="cpu")
if not isinstance(checkpoint, dict):
    raise TypeError("checkpoint 不是 state_dict")
if "pert_emb.weight" in checkpoint:
    num_perts = int(checkpoint["pert_emb.weight"].shape[0])
else:
    num_perts = max(1, len(training_targets))

model = GNN(
    total_genes=graph_adata.n_vars,
    num_perts=num_perts,
    act_fct="Sigmoid",
    hidden_channels=64,
    in_head=1,
    multi_pert=True,
).to(device)
predict_input_dim = graph_adata.n_vars + 64 + 124
model.lin_predict = DeltaPredictor(
    predict_input_dim, len(output_genes), hidden_dim=1024
).to(device)
model.load_state_dict(checkpoint)
model = model.float().eval()

print("Graph input:", graph_adata.shape)
print("Graph nodes:", int(graph.num_nodes))
print("Model output:", len(output_genes))
print("num_perts:", num_perts)
print("Checkpoint loaded:", MODEL_PATH)

In [ ]:
# =============================================================================
# VCC control → 模型输入；log-space delta → sparse raw counts
# =============================================================================

# -----------------------------------------------------------------------------
# AnnData 版本兼容读取
# 官方 controls-final 可能用 AnnData >=0.11 的 nullable-string-array 编码。
# 旧版 AnnData 遇到 obs/var index 会报 IORegistryError；这里只手动读取本流程
# 需要的 X、obs index、obs.context 和 var index，其余扩展字段可安全忽略。
# -----------------------------------------------------------------------------
def _h5_attr_str(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def _decode_h5_string_array(dataset):
    values = np.asarray(dataset)
    if values.ndim == 0:
        values = values.reshape(1)
    values = values.ravel()
    decoded = []
    for value in values:
        if isinstance(value, bytes):
            decoded.append(value.decode("utf-8"))
        else:
            decoded.append(str(value))
    return np.asarray(decoded, dtype=object).reshape(dataset.shape)


def _read_string_node(node):
    return _decode_h5_string_array(node)


def _read_nullable_string_node(node):
    if "values" not in node or "mask" not in node:
        raise KeyError("nullable-string-array 缺少 values/mask")
    values = _decode_h5_string_array(node["values"]).copy()
    mask = np.asarray(node["mask"], dtype=bool)
    if values.shape != mask.shape:
        raise ValueError("nullable-string-array 的 values 和 mask 形状不一致")
    values[mask] = None
    return values


def _read_array_node(node):
    encoding_type = _h5_attr_str(node.attrs.get("encoding-type", ""))
    if encoding_type == "string-array":
        return _read_string_node(node)
    if encoding_type == "nullable-string-array":
        return _read_nullable_string_node(node)
    if encoding_type == "categorical":
        codes = np.asarray(node["codes"]).astype(np.int64)
        categories = np.asarray(_read_array_node(node["categories"]), dtype=object).ravel()
        codes = codes.ravel()
        return pd.Categorical.from_codes(codes, categories=categories, ordered=False)
    if encoding_type == "array":
        return np.asarray(node)
    raise NotImplementedError(f"暂不支持旧版兼容读取的元素编码：{encoding_type}")


def _read_sparse_or_dense_x(node):
    encoding_type = _h5_attr_str(node.attrs.get("encoding-type", ""))
    shape = tuple(int(v) for v in np.asarray(node["shape"]).tolist()) if "shape" in node else None
    if encoding_type == "csr_matrix":
        return csr_matrix(
            (node["data"][...], node["indices"][...], node["indptr"][...]),
            shape=shape,
        )
    if encoding_type == "csc_matrix":
        return csc_matrix(
            (node["data"][...], node["indices"][...], node["indptr"][...]),
            shape=shape,
        )
    if encoding_type == "array":
        return np.asarray(node)
    raise NotImplementedError(f"暂不支持旧版兼容读取的 X 编码：{encoding_type}")


def _read_context_h5ad_legacy(path, context_hint):
    print(f"AnnData {ad.__version__} 不支持 nullable-string-array；使用兼容读取：{path.name}")
    with h5py.File(path, "r") as handle:
        if "X" not in handle:
            raise KeyError(f"{path} 缺少 X")
        X = _read_sparse_or_dense_x(handle["X"])

        if "obs" not in handle or "_index" not in handle["obs"]:
            raise KeyError(f"{path} 缺少 obs/_index")
        obs_names = np.asarray(_read_array_node(handle["obs"]["_index"]), dtype=object).ravel()
        if len(obs_names) != X.shape[0] or any(v is None for v in obs_names):
            raise ValueError("obs/_index 为空或长度与 X 不一致")
        obs_df = pd.DataFrame(index=pd.Index(obs_names.astype(str), name=None))

        if "context" in handle["obs"]:
            obs_df["context"] = np.asarray(_read_array_node(handle["obs"]["context"])).ravel()
        else:
            obs_df["context"] = context_hint

        if "var" not in handle or "_index" not in handle["var"]:
            raise KeyError(f"{path} 缺少 var/_index")
        var_names = np.asarray(_read_array_node(handle["var"]["_index"]), dtype=object).ravel()
        if len(var_names) != X.shape[1] or any(v is None for v in var_names):
            raise ValueError("var/_index 为空或长度与 X 不一致")
        var_df = pd.DataFrame(index=pd.Index(var_names.astype(str), name=None))

    return sc.AnnData(X=X, obs=obs_df, var=var_df)


def read_vcc_h5ad_compatible(path, context_hint=None):
    """新版 AnnData 走正常读取；旧版遇到 nullable-string-array 时走最小兼容读取。"""
    try:
        return sc.read(path)
    except Exception as exc:
        if "nullable-string-array" not in str(exc):
            raise
        return _read_context_h5ad_legacy(Path(path), context_hint)


graph_genes = graph_adata.var_names.astype(str).tolist()
available_graph_positions = np.array([
    i for i, gene in enumerate(graph_genes) if gene in official_gene_to_pos
], dtype=np.int32)
available_graph_official_positions = np.array([
    official_gene_to_pos[graph_genes[i]] for i in available_graph_positions
], dtype=np.int32)

node_graph_positions = []
node_official_positions = []
for node_i, graph_gene_i in enumerate(graph_gene_indices):
    gene = graph_genes[graph_gene_i]
    if gene in official_gene_to_pos:
        node_graph_positions.append(node_i)
        node_official_positions.append(official_gene_to_pos[gene])
node_graph_positions = np.array(node_graph_positions, dtype=np.int32)
node_official_positions = np.array(node_official_positions, dtype=np.int32)

print("Graph-input genes found in VCC:", len(available_graph_positions), "/", graph_adata.n_vars)
print("Graph nodes found in VCC:", len(node_graph_positions), "/", len(graph_gene_indices))


def prepare_context(context):
    """读取 raw control，构造同一批 400 细胞的图输入、全基因基线和 library size。"""
    context_path = VCC_DIR / f"context_{context}.h5ad"
    context_raw = read_vcc_h5ad_compatible(context_path, context)
    if "context" in context_raw.obs.columns:
        labels = context_raw.obs["context"].astype(str).unique().tolist()
        if len(labels) != 1:
            raise ValueError(f"{context_path} 的 context 标签不唯一：{labels}")
        context_label = labels[0]
    else:
        context_label = context

    raw_X = context_raw.X
    if issparse(raw_X):
        if not np.isfinite(raw_X.data).all() or (raw_X.data < 0).any():
            raise ValueError(f"context {context} 含非有限或负 counts")
        if not np.allclose(raw_X.data, np.rint(raw_X.data)):
            raise ValueError(f"context {context}.X 不是 raw integer counts；请检查输入层")
    else:
        raw_array = np.asarray(raw_X)
        if not np.isfinite(raw_array).all() or (raw_array < 0).any() or not np.allclose(raw_array, np.rint(raw_array)):
            raise ValueError(f"context {context}.X 不是有效的 raw integer counts")

    if context_raw.var_names.astype(str).tolist() != official_genes:
        # 仅当集合一致时自动按官方顺序重排；缺基因则直接报错。
        if set(context_raw.var_names.astype(str)) != set(official_genes):
            raise ValueError(f"context {context} 的基因面板与 gene_names.csv 不一致")
        context_raw = context_raw[:, official_genes].copy()

    context_log = context_raw.copy()
    sc.pp.normalize_total(context_log)  # 默认 target_sum=1e4，与 KOLF V4 Data 一致。
    sc.pp.log1p(context_log)

    replace = context_raw.n_obs < CELLS_PER_TARGET
    selected_indices = rng.choice(context_raw.n_obs, CELLS_PER_TARGET, replace=replace)
    selected_ids = context_raw.obs_names.astype(str)[selected_indices].tolist()
    library_sizes = np.asarray(context_raw.X[selected_indices].sum(axis=1)).ravel()
    library_sizes = np.clip(np.rint(library_sizes), 1, MAX_COUNTS_PER_CELL).astype(np.int64)

    # 400 × 18,533：全官方基因 log baseline。
    control_log_official = to_dense(context_log[selected_indices].X).astype(np.float32)

    # 400 × graph_input genes：VCC 缺失基因使用 KOLF control mean，而不是任意填零。
    control_graph_input = np.repeat(
        kolf_graph_mean_all[None, :], CELLS_PER_TARGET, axis=0
    ).astype(np.float32)
    control_graph_input[:, available_graph_positions] = to_dense(
        context_log[selected_indices, available_graph_official_positions].X
    ).astype(np.float32)

    # 当前 context 的 graph.x；VCC 缺失节点沿用 KOLF control 统计。
    context_node_values = to_dense(context_log[:, node_official_positions].X).astype(np.float32)
    node_mean = kolf_node_mean.copy()
    node_var = kolf_node_var.copy()
    node_mean[node_graph_positions] = context_node_values.mean(axis=0)
    node_var[node_graph_positions] = context_node_values.var(axis=0)
    context_graph_x = torch.from_numpy(np.stack([node_mean, node_var], axis=1)).float().to(device)

    info = {
        "context_label": context_label,
        "selected_ids": selected_ids,
        "n_available_controls": int(context_raw.n_obs),
        "sampled_with_replacement": bool(replace),
    }
    del context_raw, context_log, context_node_values
    return control_graph_input, control_log_official, library_sizes, context_graph_x, info


def make_context_target_features(context):
    """KOLF 中存在的靶点复用训练特征；缺失靶点仅用 context control 构造统计 fallback。"""
    context_path = VCC_DIR / f"context_{context}.h5ad"
    context_raw = read_vcc_h5ad_compatible(context_path, context)
    if context_raw.var_names.astype(str).tolist() != official_genes:
        context_raw = context_raw[:, official_genes].copy()
    context_log = context_raw.copy()
    sc.pp.normalize_total(context_log)
    sc.pp.log1p(context_log)

    features = {}
    feature_sources = {}
    for gene in target_genes:
        if gene in kolf_target_features:
            features[gene] = kolf_target_features[gene]
            feature_sources[gene] = "KOLF_full_control_training_feature"
        else:
            expr = to_dense(context_log[:, [gene]].X).ravel().astype(np.float32)
            mean_expr = float(expr.mean())
            log_var = float(np.log1p(max(float(expr.var()), 0.0)))
            feature = np.zeros(124, dtype=np.float32)
            feature[-4] = (mean_expr - feature_scaling["mean_center"]) / feature_scaling["mean_scale"]
            feature[-3] = (log_var - feature_scaling["logvar_center"]) / feature_scaling["logvar_scale"]
            feature[-2] = float((expr > 0).mean())
            feature[-1] = 1.0
            features[gene] = feature
            feature_sources[gene] = "VCC_context_control_stats_fallback"
    del context_raw, context_log
    return features, feature_sources


def predict_delta(target_feature, ctrl_tensor, n_rows):
    cell_type_list = [cell_type_key] * n_rows
    with torch.no_grad():
        output = model(
            {cell_type_key: graph.x},
            {cell_type_key: graph.edge_index},
            cell_type_list,
            {cell_type_key: graph.edge_index}.keys(),
            ctrl_tensor,
            target_feature.unsqueeze(0).repeat(n_rows, 1),
            {cell_type_key: graph.pos},
        )
    return output.detach().cpu().numpy().astype(np.float32)


def log_prediction_to_sparse_counts(pred_log, library_sizes):
    """逆 normalize_total+log1p，保留 top-K，并保证整数、非负、每细胞 <=1e6。"""
    if not np.isfinite(pred_log).all():
        raise FloatingPointError("模型预测含 NaN/Inf")
    pred_log = np.clip(pred_log, 0.0, 20.0)
    expected = np.expm1(pred_log) * (library_sizes[:, None].astype(np.float32) / 10000.0)
    counts = np.zeros(expected.shape, dtype=np.int32)
    k = min(MAX_GENES_PER_CELL, expected.shape[1])
    for row_i in range(expected.shape[0]):
        top = np.argpartition(expected[row_i], -k)[-k:]
        counts[row_i, top] = np.rint(expected[row_i, top]).astype(np.int64)
        total = int(counts[row_i].sum(dtype=np.int64))
        if total > MAX_COUNTS_PER_CELL:
            scaled = np.floor(
                counts[row_i].astype(np.float64) * (MAX_COUNTS_PER_CELL / total)
            )
            counts[row_i] = scaled.astype(np.int32)
    block = csr_matrix(counts)
    block.eliminate_zeros()
    block.sort_indices()
    return block

In [ ]:
# =============================================================================
# 生成 300 targets × D/E/F × 400 cells
# =============================================================================
start_time = time.time()
blocks = []
obs_records = []
target_report_rows = []
run_context_info = {}
row_cursor = 0
total_cells = len(CONTEXTS) * len(target_genes) * CELLS_PER_TARGET
max_possible_stored = total_cells * min(MAX_GENES_PER_CELL, len(official_genes))
if max_possible_stored > MAX_STORED_ENTRIES:
    raise ValueError("当前 MAX_GENES_PER_CELL 可能超过官方存储密度上限，请调小")

for context in CONTEXTS:
    print(f"\n================ Context {context} ================")
    (
        control_graph_input,
        control_log_official,
        library_sizes,
        context_graph_x,
        context_info,
    ) = prepare_context(context)
    run_context_info[context] = context_info
    graph.x = context_graph_x
    ctrl_tensor = torch.from_numpy(control_graph_input).float().to(device)
    context_features, feature_sources = make_context_target_features(context)

    for target_index, target_gene in enumerate(target_genes, start=1):
        target_feature = torch.from_numpy(context_features[target_gene]).float().to(device)
        pred_delta = np.empty((CELLS_PER_TARGET, len(output_genes)), dtype=np.float32)
        for start in range(0, CELLS_PER_TARGET, PREDICT_BATCH_SIZE):
            end = min(start + PREDICT_BATCH_SIZE, CELLS_PER_TARGET)
            pred_delta[start:end] = predict_delta(
                target_feature, ctrl_tensor[start:end], end - start
            )

        # 未覆盖的 653 个官方基因 delta=0；覆盖基因使用 VCC control + KOLF delta。
        pred_log = control_log_official.copy()
        pred_log[:, output_official_positions] = np.maximum(
            control_log_official[:, output_official_positions] + pred_delta, 0.0
        )
        block = log_prediction_to_sparse_counts(pred_log, library_sizes)
        blocks.append(block)

        context_label = context_info["context_label"]
        for local_i in range(CELLS_PER_TARGET):
            obs_records.append({
                "obs_id": f"{context_label}__{target_gene}__{local_i:03d}",
                "target_gene": target_gene,
                "context": context_label,
            })
        row_cursor += CELLS_PER_TARGET
        target_report_rows.append({
            "context": context_label,
            "target_gene": target_gene,
            "feature_source": feature_sources[target_gene],
            "in_kolf_output_genes": target_gene in output_gene_set,
            "in_kolf_direct_training": target_gene in set(training_targets),
            "in_kolf_ood_split": target_gene in set(testing_genes),
            "predicted_stored_entries": int(block.nnz),
            "max_delta_abs": float(np.max(np.abs(pred_delta))),
            "mean_delta": float(np.mean(pred_delta)),
        })
        if target_index % 25 == 0 or target_index == len(target_genes):
            print(
                f"context {context}: {target_index}/{len(target_genes)} targets; "
                f"rows={row_cursor:,}; elapsed={(time.time()-start_time)/60:.1f} min"
            )

    del control_graph_input, control_log_official, ctrl_tensor, context_graph_x

if row_cursor != total_cells:
    raise RuntimeError(f"行数错误：{row_cursor} != {total_cells}")
print("Concatenating sparse blocks ...")
X_pred = vstack(blocks, format="csr", dtype=np.int32)
X_pred.eliminate_zeros()
X_pred.sort_indices()
del blocks

obs_df = pd.DataFrame(obs_records).set_index("obs_id")
obs_df["target_gene"] = pd.Categorical(obs_df["target_gene"], categories=target_genes, ordered=True)
obs_df["context"] = pd.Categorical(
    obs_df["context"],
    categories=[run_context_info[c]["context_label"] for c in CONTEXTS],
    ordered=True,
)
var_df = pd.DataFrame(index=pd.Index(official_genes, name=None))
prediction = sc.AnnData(X=X_pred, obs=obs_df, var=var_df)
target_report_df = pd.DataFrame(target_report_rows)

print("Prediction:", prediction.shape)
print("Stored entries:", prediction.X.nnz)
print("Elapsed minutes:", round((time.time() - start_time) / 60, 1))

In [ ]:
# =============================================================================
# 本地硬性格式检查 + 保存 H5AD / 可复现性记录
# =============================================================================
expected_context_labels = [run_context_info[c]["context_label"] for c in CONTEXTS]
assert prediction.shape == (total_cells, 18533)
assert prediction.var_names.astype(str).tolist() == official_genes
assert prediction.obs_names.is_unique
assert set(prediction.obs["context"].astype(str)) == set(expected_context_labels)
assert set(prediction.obs["target_gene"].astype(str)) == set(target_genes)
assert "non-targeting" not in set(prediction.obs["target_gene"].astype(str))
assert prediction.X.nnz <= MAX_STORED_ENTRIES
assert np.isfinite(prediction.X.data).all()
assert (prediction.X.data >= 0).all()
assert np.allclose(prediction.X.data, np.rint(prediction.X.data))

counts_per_group = prediction.obs.groupby(
    ["context", "target_gene"], observed=True
).size()
assert len(counts_per_group) == len(CONTEXTS) * len(target_genes)
assert (counts_per_group == CELLS_PER_TARGET).all()
row_sums = np.asarray(prediction.X.sum(axis=1)).ravel()
assert row_sums.max() <= MAX_COUNTS_PER_CELL

print("Shape / official gene order: OK")
print("Exact target and context sets: OK")
print("400 cells per target/context: OK")
print("Raw non-negative integer counts: OK")
print("Sparse stored-entry cap: OK")
print("Per-cell count cap: OK")
print("Max counts in one cell:", int(row_sums.max()))
print("Median counts per cell:", float(np.median(row_sums)))
print("Stored entries:", int(prediction.X.nnz))

# 写盘前不覆盖旧文件以外的任何训练资产；允许重复运行时覆盖本预测输出。
prediction.write(OUTPUT_H5AD, compression="gzip")
target_report_df.to_csv(TARGET_REPORT_PATH, index=False)

run_manifest = {
    "version": "V4_KOLF_to_VCC2026",
    "model_path": str(MODEL_PATH),
    "graph_input_path": str(GRAPH_INPUT_PATH),
    "full_input_path": str(FULL_INPUT_PATH),
    "output_gene_order_path": str(OUTPUT_ORDER_PATH),
    "prediction_h5ad": str(OUTPUT_H5AD),
    "shape": list(prediction.shape),
    "contexts": expected_context_labels,
    "n_targets": len(target_genes),
    "cells_per_target_context": CELLS_PER_TARGET,
    "random_seed": RANDOM_SEED,
    "predict_batch_size": PREDICT_BATCH_SIZE,
    "max_genes_per_cell": MAX_GENES_PER_CELL,
    "stored_entries": int(prediction.X.nnz),
    "model_output_genes": len(output_genes),
    "official_genes_delta_zero": len(official_genes) - len(output_genes),
    "target_feature_fallback_rows": int((target_report_df.feature_source == "VCC_context_control_stats_fallback").sum()),
    "context_control_sampling": run_context_info,
}
with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)

print("Saved H5AD:", OUTPUT_H5AD)
print("Saved target report:", TARGET_REPORT_PATH)
print("Saved run manifest:", RUN_MANIFEST_PATH)

In [ ]:
# =============================================================================
# 官方 CLI：校验并打包为 .vcc（最终阶段请使用 vcc >= 0.3.0）
# 不上传；上传由你在官网或自行执行 vcc submit 完成。
# =============================================================================
prep_command = [
    "vcc", "prep", str(OUTPUT_H5AD),
    "-g", str(GENE_NAMES_PATH),
    "--perts", str(PERT_COUNTS_PATH),
    "--context-col", "context",
    "--contexts", ",".join(expected_context_labels),
    "-o", str(OUTPUT_VCC),
    "-f",
]
print("Packaging command:")
print(subprocess.list2cmdline(prep_command))

if RUN_VCC_PREP:
    if shutil.which("vcc") is None:
        raise FileNotFoundError("未找到 vcc 命令；请先安装/升级官方 VCC CLI（最终阶段 >=0.3.0）")
    subprocess.run(["vcc", "--version"], check=True)
    subprocess.run(prep_command, check=True)
    print("Created:", OUTPUT_VCC)
    print("可提交文件已生成；Notebook 不会自动上传。")
else:
    print("RUN_VCC_PREP=False：目前只生成并验证 H5AD。确认后改为 True，单独运行本单元。")